# Fine-Tune Gemma 3 12B for Legal Partner

Fine-tunes `google/gemma-3-12b-it` on 1,827 legal task examples using QLoRA.

**Requirements:** A100 40GB GPU, ~6-10 hours training time.

**Tasks trained:** Drafting, Risk Assessment, Extraction, Checklist, Redline

In [ ]:
# Cell 1 - Check GPU
!nvidia-smi
# Must show A100 40GB

In [ ]:
# Cell 2 - Install dependencies
!pip install --upgrade transformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets huggingface_hub

In [ ]:
# Cell 3 - HuggingFace login
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # https://huggingface.co/settings/tokens

In [ ]:
# Cell 4 - Mount Drive + Clone training data
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/jyoti0512shukla/legal-finetune.git /content/legal-finetune 2>/dev/null || echo 'Already cloned'
!cd /content/legal-finetune && git checkout v3
!wc -l /content/legal-finetune/data/gemma4/train.jsonl
!wc -l /content/legal-finetune/data/gemma4/validation.jsonl

In [ ]:
# Cell 5 - Load Gemma 3 12B with LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "google/gemma-3-12b-it",
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

model.print_trainable_parameters()

In [ ]:
# Cell 6 - Load and format training data
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="gemma")

dataset = load_dataset("json",
    data_files="/content/legal-finetune/data/gemma4/train.jsonl",
    split="train")

def format_chat(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False
            )
            for c in examples["conversations"]
        ]
    }

train_data = dataset.map(format_chat, batched=True)
print(f"Training examples: {len(train_data)}")
print(f"\nSample:\n{train_data[0]['text'][:500]}")

In [ ]:
# Cell 7 - Train (~6-10 hours on A100)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_data,
    dataset_text_field = "text",
    max_seq_length     = 2048,
    args = TrainingArguments(
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 16,
        num_train_epochs             = 3,
        learning_rate                = 1e-4,
        fp16                         = not torch.cuda.is_bf16_supported(),
        bf16                         = torch.cuda.is_bf16_supported(),
        logging_steps                = 25,
        save_steps                   = 200,
        save_total_limit             = 2,
        optim                        = "adamw_8bit",
        lr_scheduler_type            = "cosine",
        warmup_steps                 = 17,
        weight_decay                 = 0.01,
        output_dir                   = "/content/drive/MyDrive/gemma3-legal-checkpoints",
    ),
)

trainer_stats = trainer.train()
print(f"\nDone. Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Cell 8 - Validate
FastLanguageModel.for_inference(model)

prompts = [
    ("DRAFT", "Draft a termination clause for a vendor agreement governed by Indian law. Write exactly 4 numbered sub-clauses as plain legal prose."),
    ("RISK", "Assess the risk level of this clause:\n\n'The vendor shall not be liable for any damages whatsoever arising from this agreement, whether direct, indirect, or consequential.'"),
    ("EXTRACT", "Extract key terms from this clause:\n\n'This Agreement between Acme Pvt Ltd (Vendor) and Beta Corp (Client), effective 1 January 2026, for a term of 2 years at INR 24,00,000 per annum, governed by the laws of India, with arbitration in Mumbai.'"),
]

for label, prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    result = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}\n{label}\n{'='*60}\n{result[:600]}\n")

# CHECK:
# DRAFT   -> plain prose numbered clauses, NO JSON
# RISK    -> structured risk output with level + issues
# EXTRACT -> labelled key terms

In [ ]:
# Cell 9 - Save adapter to HuggingFace
model.push_to_hub("jyoti0512shuklaorg/gemma3-legal-v1", private=True)
tokenizer.push_to_hub("jyoti0512shuklaorg/gemma3-legal-v1", private=True)
print("Adapter saved")

In [ ]:
# Cell 10 - Merge into full model
model.save_pretrained_merged(
    "/content/drive/MyDrive/gemma3-legal-v1-merged",
    tokenizer,
    save_method="merged_16bit",
)
print("Merged model saved to Drive")

model.push_to_hub_merged(
    "jyoti0512shuklaorg/gemma3-legal-v1-merged",
    tokenizer,
    save_method="merged_16bit",
    private=True,
)
print("Merged model pushed to HuggingFace")

In [ ]:
# Cell 11 - Export GGUF for customer L4 VMs
model.push_to_hub_gguf(
    "jyoti0512shuklaorg/gemma3-legal-v1-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    private=True,
)
print("GGUF pushed to HuggingFace")

In [ ]:
# Cell 12 - Test with ngrok (optional - connect to your VM)
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_TOKEN")
url = ngrok.connect(8000)
print(f"\nLEGALPARTNER_CHAT_API_URL={url.public_url}/v1")
print(f"LEGALPARTNER_CHAT_API_MODEL=jyoti0512shuklaorg/gemma3-legal-v1-merged")

!vllm serve /content/drive/MyDrive/gemma3-legal-v1-merged \
  --port 8000 --host 0.0.0.0 \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.90 \
  --max-num-seqs 5